<a href="https://colab.research.google.com/github/sajid-shahriar/pytorch-tutorial/blob/main/ltb_build_the_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

print(f"Using {device} device")

Using cuda device


#Define the class

We define our neural network by subclassing nn.Module, and initialize the neural network layers in __init__. Every nn.Module subclass implements the operations on input data in the forward method.


In [3]:
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()

    self.linear_relu_stack = nn.Sequential(
        nn.Linear(28 * 28, 512),
        nn.ReLU(),
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Linear(512,10)
    )

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits


In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


**Do not call model.forward() directly. To use the model pass the input data**

In [5]:
# this code is experimental. wanted to know
# how softmax works

m = nn.Softmax(dim=1)
input = torch.randn(2, 3)
output = m(input)
print(input)
print(output)

tensor([[ 0.0343, -1.3436, -0.2278],
        [-0.9767,  0.7449,  1.4030]])
tensor([[0.4947, 0.1247, 0.3806],
        [0.0575, 0.3215, 0.6210]])


In [6]:
x = torch.rand(3, 28, 28, device = device)
logits = model(x)
pred_probability = nn.Softmax(dim=1)(logits)

y_pred = pred_probability.argmax(1)

# print(f"logits: {logits}")
print(f"logits shape: {logits.shape}")
# print(f"pred_probability: {pred_probability}")
print(f"pred_probability shape: {pred_probability.shape}")
print(f"y_pred: {y_pred}")
print(f"y_pred shape: {y_pred.shape}")

logits shape: torch.Size([3, 10])
pred_probability shape: torch.Size([3, 10])
y_pred: tensor([3, 3, 3], device='cuda:0')
y_pred shape: torch.Size([3])


#Model layers


In [7]:
input_image = torch.rand(3,28,28)
print(input_image.shape)

torch.Size([3, 28, 28])


In [8]:
# nn.Flatten()
# Flattens a contiguous range of dims into a tensor.
# start_dim (default = 1) and end_dim(default = -1)

flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.shape)

torch.Size([3, 784])


In [9]:
# nn.Linear
# Applies linear transformation a_1 = W a_0 + b

layer_1 = nn.Linear(
    in_features = 28 * 28,
    out_features = 20
)

hidden_1 = layer_1(flat_image)
print(hidden_1.shape)

torch.Size([3, 20])


In [10]:
# nn.ReLU : Rectified linear unit :  returns max(value, 0)

print(f"before ReLU: \n{hidden_1}\n")
hidden_1 = nn.ReLU()(hidden_1)
print(f"after ReLU: \n{hidden_1}\n")

before ReLU: 
tensor([[-0.0691, -0.2805, -0.3973,  0.1229,  0.2745, -0.1398, -0.0924,  0.1980,
          0.1560, -0.8404, -0.1697,  0.6636,  0.2978, -0.0343, -0.2887,  0.4509,
          0.1208, -0.1042, -0.0309, -0.3430],
        [ 0.1129, -0.2172, -0.2775,  0.1359,  0.1711, -0.1499,  0.0877, -0.3642,
         -0.1573, -0.7432, -0.1763,  0.6602, -0.0623,  0.0342, -0.7609,  0.3040,
         -0.0398,  0.2270, -0.2727, -0.4500],
        [ 0.0445,  0.0690, -0.0292,  0.4256, -0.0873, -0.2556, -0.1472, -0.0718,
          0.2381, -0.5169,  0.1266,  0.6459,  0.1000,  0.0425, -0.4432,  0.1932,
          0.1284,  0.1400, -0.0573, -0.0649]], grad_fn=<AddmmBackward0>)

after ReLU: 
tensor([[0.0000, 0.0000, 0.0000, 0.1229, 0.2745, 0.0000, 0.0000, 0.1980, 0.1560,
         0.0000, 0.0000, 0.6636, 0.2978, 0.0000, 0.0000, 0.4509, 0.1208, 0.0000,
         0.0000, 0.0000],
        [0.1129, 0.0000, 0.0000, 0.1359, 0.1711, 0.0000, 0.0877, 0.0000, 0.0000,
         0.0000, 0.0000, 0.6602, 0.0000, 0.0342, 0.0

In [11]:
# nn.Sequential
# Ordered container of modules.
# the data is passes through all the modules
# in th same order as defined

seq_modules = nn.Sequential(
    flatten,
    layer_1,
    nn.ReLU(),
    nn.Linear(20,10)
)

input_image = torch.rand(3,28,28)

logits = seq_modules(input_image)

print(f"shape of logits: {logits.shape}")


shape of logits: torch.Size([3, 10])


In [12]:
softmax = nn.Softmax(dim=1)

pred_probability = softmax(logits)

print(f"pred_probability: {pred_probability}")
print(f"pred_probability shape: {pred_probability.shape}")

pred_probability: tensor([[0.0912, 0.1086, 0.0977, 0.0992, 0.1422, 0.0635, 0.0957, 0.0962, 0.0836,
         0.1222],
        [0.0947, 0.1010, 0.1065, 0.0970, 0.1442, 0.0550, 0.0964, 0.1037, 0.0772,
         0.1244],
        [0.0840, 0.1163, 0.0850, 0.1092, 0.1345, 0.0664, 0.1039, 0.1077, 0.0812,
         0.1118]], grad_fn=<SoftmaxBackward0>)
pred_probability shape: torch.Size([3, 10])


In [13]:
print(f"Model structure: {model}\n\n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)




In [14]:
for name, param in model.named_parameters():
  print(f"Layer: {name} | Size: {param.shape}")
  # print(f"Params values: {param}")

Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784])
Layer: linear_relu_stack.0.bias | Size: torch.Size([512])
Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512])
Layer: linear_relu_stack.2.bias | Size: torch.Size([512])
Layer: linear_relu_stack.4.weight | Size: torch.Size([10, 512])
Layer: linear_relu_stack.4.bias | Size: torch.Size([10])
